# 02 - PhoBERT Sentiment & Annotation Pipeline

This notebook acts as an execution and inspection runner for the sentiment analysis and article annotation pipeline.

When running on Kaggle, this notebook will:
- Read keys (`GITHUB_TOKEN`, `GEMINI_API_KEY`, etc.) from Kaggle Secrets.
- Clone or pull the active branch of the GitHub repo into `/kaggle/working`.
- Materialize credentials in `.dvc/credentials/key.json` and `.dvc/config.local` for service account access.
- Pull DVC tracking targets (like raw news files).
- Install dependencies and run pipeline stages against the checked-out workspace.

## 0 - Bootstrap

In [1]:
import base64
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

STAGED_ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
IS_KAGGLE = Path('/kaggle').exists()

def run_command(*args: str, cwd: str | Path | None = None, display: str | None = None) -> None:
    command = [str(arg) for arg in args]
    cmd_str = display or ' '.join(command)
    import re
    cmd_str = re.sub(r'https://[^@]+@github\.com', 'https://<TOKEN>@github.com', cmd_str)
    print('$', cmd_str)
    subprocess.run(command, check=True, cwd=cwd)

def load_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding='utf-8'))

def load_kaggle_secret(name: str) -> str | None:
    if not IS_KAGGLE:
        return os.getenv(name)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.getenv(name)

def parse_json_secret(value: str | None) -> dict | None:
    if not value:
        return None
    candidates = [value]
    try:
        decoded = base64.b64decode(value).decode('utf-8')
        candidates.append(decoded)
    except Exception:
        pass
    for candidate in candidates:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            continue
    return None

def write_service_account_key(secret_name: str, repo_root: Path) -> Path | None:
    secret_payload = parse_json_secret(load_kaggle_secret(secret_name))
    if not secret_payload:
        return None
    key_path = repo_root / '.dvc' / 'credentials' / 'key.json'
    key_path.parent.mkdir(parents=True, exist_ok=True)
    key_path.write_text(json.dumps(secret_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return key_path

def configure_dvc_service_account(repo_root: Path, key_path: Path, remote_name: str = 'gdrive') -> Path:
    config_local_path = repo_root / '.dvc' / 'config.local'
    rel_key_path = key_path.relative_to(repo_root / '.dvc')
    config_lines = [
        f'[remote "{remote_name}"]',
        '    gdrive_use_service_account = true',
        f'    gdrive_service_account_json_file_path = {rel_key_path.as_posix()}'
    ]
    config_local_path.write_text('\n'.join(config_lines) + '\n', encoding='utf-8')
    return config_local_path

def github_auth_url(repo_url: str, token: str | None) -> str:
    if not token or not repo_url.startswith('https://github.com/'):
        return repo_url
    return repo_url.replace('https://', f'https://{token}@', 1)

print('Running on Kaggle:', IS_KAGGLE)
print('Staged root     :', STAGED_ROOT)


Running on Kaggle: True
Staged root     : /kaggle/working


## 1 - Runtime Configuration

In [2]:
REPO_GIT_URL = 'https://github.com/tlong-ds/news-sentiment-analysis.git'
REPO_BRANCH = 'main'
REPO_DIR_NAME = 'news-sentiment-analysis'
PULL_REPO_ON_KAGGLE = IS_KAGGLE
INSTALL_PROJECT_REQUIREMENTS = IS_KAGGLE
LOAD_KAGGLE_SECRETS = IS_KAGGLE
GDRIVE_SERVICE_ACCOUNT_SECRET = 'GDRIVE_SERVICE_ACCOUNT_JSON'
SETUP_DVC_SERVICE_ACCOUNT = IS_KAGGLE
DVC_PULL_ON_KAGGLE = IS_KAGGLE
DVC_PULL_REMOTE = 'gdrive'
DVC_PULL_TARGETS = ['data/main/raw.dvc']


## 2 - Kaggle Secrets, Repo Pull, and DVC Bootstrap

In [3]:
if LOAD_KAGGLE_SECRETS:
    for secret_name in ['GITHUB_TOKEN', 'GEMINI_API_KEY']:
        secret_value = load_kaggle_secret(secret_name)
        if secret_value and secret_name not in os.environ:
            os.environ[secret_name] = secret_value

if IS_KAGGLE and PULL_REPO_ON_KAGGLE:
    kaggle_repo_root = Path('/kaggle/working') / REPO_DIR_NAME
    auth_repo_url = github_auth_url(REPO_GIT_URL, os.getenv('GITHUB_TOKEN'))

    if kaggle_repo_root.exists() and (kaggle_repo_root / '.git').exists():
        run_command('git', 'fetch', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'checkout', REPO_BRANCH, cwd=kaggle_repo_root)
        run_command('git', 'pull', 'origin', REPO_BRANCH, cwd=kaggle_repo_root)
    else:
        run_command(
            'git', 'clone', '--branch', REPO_BRANCH, auth_repo_url, str(kaggle_repo_root),
            cwd='/kaggle/working'
        )
    ROOT = kaggle_repo_root
else:
    ROOT = STAGED_ROOT

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
importlib.invalidate_caches()

if INSTALL_PROJECT_REQUIREMENTS:
    run_command(sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', cwd=ROOT)
else:
    print('Skipping pip install.')

service_account_key_path = None
dvc_config_local_path = None
if SETUP_DVC_SERVICE_ACCOUNT:
    service_account_key_path = write_service_account_key(GDRIVE_SERVICE_ACCOUNT_SECRET, ROOT)
    if service_account_key_path is None:
        print(f'{GDRIVE_SERVICE_ACCOUNT_SECRET} not found or invalid; skipping DVC service-account setup.')
    else:
        dvc_config_local_path = configure_dvc_service_account(ROOT, service_account_key_path, remote_name=DVC_PULL_REMOTE)
        os.environ.setdefault('DVC_SITE_CACHE_DIR', str(ROOT / '.dvc' / 'site-cache'))
        if DVC_PULL_ON_KAGGLE:
            run_command('dvc', 'pull', '-r', DVC_PULL_REMOTE, *DVC_PULL_TARGETS, cwd=ROOT)
else:
    print('SETUP_DVC_SERVICE_ACCOUNT is False; skipping DVC auth bootstrap.')

# Import path constants dynamically after project root is setup
from src.config import CAFEF_DATA_DIR, MODELS_DATA_DIR, PROCESSED_DATA_DIR

# Stage 1 - Prepare training corpus (train on full_data.csv)
FULL_DATA_CSV = ROOT / 'data' / 'main' / 'raw' / 'full_data.csv'
TRAINING_OUTPUT = ROOT / 'data' / 'main' / 'sentiment' / 'training_corpus.parquet'

# Stage 2 - Bootstrap labels (LLM-assisted labeling)
BOOTSTRAP_OUTPUT = ROOT / 'data' / 'main' / 'sentiment' / 'training_bootstrap_labels.parquet'
MERGED_LABELED_OUTPUT = ROOT / 'data' / 'main' / 'sentiment' / 'training_labeled.parquet'

# Stage 3 - Train classifier (model artifacts + hyperparameters)
MODEL_DIR = ROOT / 'models' / 'phobert-sentiment' / 'latest'
BASE_MODEL = os.getenv('PHOBERT_BASE_MODEL', 'vinai/phobert-base-v2')
BATCH_SIZE = 16
MAX_LENGTH = 256
EPOCHS = 2
LEARNING_RATE = 2e-5
SEED = 42

# Stage 4 - Inference + validation (CafeF articles only)
CAFEF_ARTICLES = ROOT / 'data' / 'main' / 'processed' / 'articles_clean.parquet'
CAFEF_INPUT = ROOT / 'data' / 'main' / 'cafef' / 'cafef_input.parquet'
SENTIMENT_OUTPUT = ROOT / 'data' / 'main' / 'processed' / 'article_sentiment_scores.parquet'
DAILY_NEWS = ROOT / 'data' / 'main' / 'processed' / 'daily_news_prices.parquet'

# LLM Bootstrapping configurations
BOOTSTRAP_BACKEND = 'gemini'
BOOTSTRAP_MODEL = 'gemini-2.0-flash'
CONFIDENCE_THRESHOLD = 0.8

def run_module(module: str, *args: str) -> None:
    run_command(sys.executable, '-m', module, *args, cwd=ROOT)

for path in [TRAINING_OUTPUT.parent, MODEL_DIR.parent, SENTIMENT_OUTPUT.parent]:
    path.mkdir(parents=True, exist_ok=True)


$ git clone --branch main https://<TOKEN>@github.com/tlong-ds/news-sentiment-analysis.git /kaggle/working/news-sentiment-analysis


Cloning into '/kaggle/working/news-sentiment-analysis'...


$ /usr/bin/python3 -m pip install -q -r requirements.txt
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.7/279.7 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.3 MB/s eta

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.


$ dvc pull -r gdrive data/main/raw.dvc
A       data/main/raw/
7 files fetched and 6 files added


## 3 - Prepare A Normalized Training Corpus

In [4]:
if FULL_DATA_CSV.exists():
    print('Preparing training corpus from FULL_DATA_CSV (full_data.csv).')
    run_module(
        'src.sentiment.prepare_training_data',
        '--extra-input', str(FULL_DATA_CSV),
        '--extra-source-name', 'full_data',
        '--output-file', str(TRAINING_OUTPUT),
    )
else:
    raise FileNotFoundError(
        f'Missing training source. Expected {FULL_DATA_CSV}.'
    )

pd.read_parquet(TRAINING_OUTPUT).head(3)


Preparing training corpus from FULL_DATA_CSV (full_data.csv).
$ /usr/bin/python3 -m src.sentiment.prepare_training_data --extra-input /kaggle/working/news-sentiment-analysis/data/main/raw/full_data.csv --extra-source-name full_data --output-file /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_corpus.parquet


2026-05-26 09:02:33,424 - INFO - Prepared 107200 training rows -> /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_corpus.parquet


,article_id,source,category,published_at,title,body_text,input_text,source_dataset
0,https://baodautu.vn/nam-dinh-thanh-lap-cum-con...,full_data,Đầu tư,2024-12-31 00:00:00,Nam Định thành lập Cụm công nghiệp Mỹ Tân vốn ...,"Với tổng mứcđầu tưlên tới 266 tỷ đồng, CCN Mỹ ...",Nam Định thành lập Cụm công nghiệp Mỹ Tân vốn ...,full_data
1,https://baodautu.vn/nganh-tai-chinh-se-tao-die...,full_data,Tài chính - Chứng khoán,2024-12-31 00:00:00,Ngành tài chính sẽ tạo điều kiện huy động mọi ...,"Chiều 31/12, Thủ tướng Chính phủ Phạm Minh Chí...",Ngành tài chính sẽ tạo điều kiện huy động mọi ...,full_data
2,https://baodautu.vn/doanh-nghiep-thanh-pho-tha...,full_data,Doanh nghiệp,2024-12-31 00:00:00,Doanh nghiệp thành phố Thái Bình hướng về ngườ...,"Trong những năm qua, cộng đồngdoanh nghiệpvàdo...",Doanh nghiệp thành phố Thái Bình hướng về ngườ...,full_data


## 4 - LLM Bootstrapping Stage (Ollama / Gemini)

Uses local or hosted language models to tag expected market-impact sentiment for training corpus articles. Make sure `GEMINI_API_KEY` is present in your environment variables if using the `gemini` backend.

In [5]:
if BOOTSTRAP_BACKEND == 'gemini':
    assert os.getenv('GEMINI_API_KEY'), "GEMINI_API_KEY environment variable is required for Gemini bootstrapping."

run_module(
    'src.sentiment.bootstrap_labels',
    '--input-file', str(TRAINING_OUTPUT),
    '--output-file', str(BOOTSTRAP_OUTPUT),
    '--backend', BOOTSTRAP_BACKEND,
    '--model', BOOTSTRAP_MODEL,
    '--confidence-threshold', str(CONFIDENCE_THRESHOLD),
)

# Merge LLM bootstrap annotations back into training corpus splits
run_module(
    'src.sentiment.merge_annotations',
    '--corpus-file', str(TRAINING_OUTPUT),
    '--annotations-file', str(BOOTSTRAP_OUTPUT),
    '--output-file', str(MERGED_LABELED_OUTPUT),
    '--confidence-threshold', str(CONFIDENCE_THRESHOLD),
    '--seed', str(SEED),
)

pd.read_parquet(MERGED_LABELED_OUTPUT)[['article_id', 'label', 'confidence', 'split']].head(5)


$ /usr/bin/python3 -m src.sentiment.bootstrap_labels --input-file /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_corpus.parquet --output-file /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_bootstrap_labels.parquet --backend gemini --model gemini-2.0-flash --confidence-threshold 0.8


2026-05-26 13:32:39,407 - INFO - Bootstrapped 107200 labels -> /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_bootstrap_labels.parquet


$ /usr/bin/python3 -m src.sentiment.merge_annotations --corpus-file /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_corpus.parquet --annotations-file /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_bootstrap_labels.parquet --output-file /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_labeled.parquet --confidence-threshold 0.8 --seed 42


2026-05-26 13:32:51,068 - INFO - Merged 36053 labeled rows -> /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_labeled.parquet


,article_id,label,confidence,split
0,https://baodautu.vn/10-thang-nen-kinh-te-co-th...,positive,0.90,test
1,https://baodautu.vn/17-trieu-san-pham-cua-doan...,positive,0.80,test
2,https://baodautu.vn/1c-viet-nam-va-aed-chung-t...,positive,0.80,test
3,https://baodautu.vn/2000-dai-bieu-tham-du-hoi-...,positive,0.80,test
4,https://baodautu.vn/330-doanh-nghiep-nuoc-ngoa...,positive,0.85,test


## 5 - Train The Classifier

In [6]:
run_module(
    'src.sentiment.train_classifier',
    '--labeled-input', str(MERGED_LABELED_OUTPUT),
    '--output-dir', str(MODEL_DIR),
    '--base-model', str(BASE_MODEL),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--max-length', str(MAX_LENGTH),
    '--seed', str(SEED),
)
pd.read_json(MODEL_DIR / 'evaluation.json')


$ /usr/bin/python3 -m src.sentiment.train_classifier --labeled-input /kaggle/working/news-sentiment-analysis/data/main/sentiment/training_labeled.parquet --output-dir /kaggle/working/news-sentiment-analysis/models/phobert-sentiment/latest --base-model vinai/phobert-base-v2 --epochs 2 --batch-size 16 --learning-rate 2e-05 --max-length 256 --seed 42


2026-05-26 13:33:08,293 - INFO - Using CUDA GPU device: Tesla T4
2026-05-26 13:33:08,395 - INFO - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-26 13:33:08,403 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/vinai/phobert-base-v2/e2375d266bdf39c6e8e9a87af16a5da3190b0cc8/config.json "HTTP/1.1 200 OK"
2026-05-26 13:33:08,411 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/vinai/phobert-base-v2/e2375d266bdf39c6e8e9a87af16a5da3190b0cc8/config.json "HTTP/1.1 200 OK"
2026-05-26 13:33:08,438 - INFO - HTTP Request: HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 404 Not Found"
2026-05-26 13:33:08,438 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-26 13:33:08,475 - INFO - HTTP Request: HEAD https://hug

,train,test,val,test_by_source
rows,25725,10328,0,NaN
accuracy,0.543168,0.971534,None,NaN
label_distribution,"{'positive': 16786, 'negative': 8752, 'neutral...","{'positive': 6716, 'negative': 3541, 'neutral'...",{},NaN
predicted_distribution,"{'positive': 16963, 'negative': 8696, 'neutral...","{'positive': 6812, 'negative': 3502, 'neutral'...",{},NaN
confusion_matrix,"{'negative': {'negative': 2936, 'neutral': 22,...","{'negative': {'negative': 3395, 'neutral': 0, ...",NaN,NaN
skipped_training,NaN,NaN,True,NaN
full_data,NaN,NaN,NaN,"{'rows': 10328, 'accuracy': 0.9715336948102241..."


## 6 - Run CafeF Inference And Validation

In [7]:
run_module(
    'src.preprocessing.pipeline',
    '--raw-news',
    'data/main/raw/news_VN_cafef.csv',
    '--prices', 'data/main/raw/prices_VN.csv',
    '--out-dir', 'data/main/processed'
)

run_module(
    'src.sentiment.run_pipeline',
    '--mode', 'infer',
    '--model-dir', str(MODEL_DIR),
    '--cafef-input', str(CAFEF_ARTICLES),
    '--cafef-prepared-output', str(CAFEF_INPUT),
    '--sentiment-output', str(SENTIMENT_OUTPUT),
    '--daily-news-file', str(DAILY_NEWS),
    '--batch-size', str(BATCH_SIZE),
    '--max-length', str(MAX_LENGTH),
)
pd.read_parquet(SENTIMENT_OUTPUT).head(5)


$ /usr/bin/python3 -m src.preprocessing.pipeline --raw-news data/main/raw/news_VN_cafef.csv --prices data/main/raw/prices_VN.csv --out-dir data/main/processed


<frozen runpy>:128: RuntimeWarning: 'src.preprocessing.pipeline' found in sys.modules after import of package 'src.preprocessing', but prior to execution of 'src.preprocessing.pipeline'; this may result in unpredictable behaviour
14:19:21 INFO Loading raw news: data/main/raw/news_VN_cafef.csv
14:19:34 INFO Raw news loaded: 86234 rows, columns: ['url', 'source', 'category', 'title', 'date', 'published_at', 'body']
14:19:34 INFO Cleaning article body text …
14:21:03 INFO Short-article filter (<100 chars): 47 removed, 86187 kept
14:21:03 INFO Aligning 86187 articles to trading days …
14:21:07 WARNING 8 articles could not be mapped to a trading day (unmapped) — dropping.
14:21:07 INFO Aggregating daily news controls …
14:21:07 INFO Merging daily news with price calendar …
14:21:07 INFO Preprocessing complete — 86179 articles, 2498 daily rows.
14:21:07 INFO Writing articles_clean.parquet (86179 rows) …
14:21:15 INFO Writing daily_news_prices.parquet (2498 rows) …
14:21:15 INFO Writing prepr

{
  "articles_clean": "/kaggle/working/news-sentiment-analysis/data/main/processed/articles_clean.parquet",
  "daily_news_prices": "/kaggle/working/news-sentiment-analysis/data/main/processed/daily_news_prices.parquet",
  "diagnostics": "/kaggle/working/news-sentiment-analysis/data/main/processed/preprocessing_diagnostics.json"
}
{
  "generated_at": "2026-05-26T14:21:07",
  "raw_news_path": "/kaggle/working/news-sentiment-analysis/data/main/raw/news_VN_cafef.csv",
  "price_path": "/kaggle/working/news-sentiment-analysis/data/main/raw/prices_VN.csv",
  "raw_cafef_row_count": 86234,
  "after_short_filter_row_count": 86187,
  "short_articles_removed": 47,
  "min_body_len_threshold": 100,
  "unmapped_articles_dropped": 8,
  "cleaned_article_row_count": 86179,
  "published_at_non_null_share": 1.0,
  "timestamp_based_alignment_share": 1.0,
  "date_only_fallback_share": 0.0,
  "after_close_forward_shifts": 25041,
  "non_trading_day_forward_shifts": 16563,
  "processed_daily_row_count": 2498,


2026-05-26 14:21:21,356 - INFO - Running: /usr/bin/python3 -m src.sentiment.prepare_inputs --cafef-input /kaggle/working/news-sentiment-analysis/data/main/processed/articles_clean.parquet --cafef-output /kaggle/working/news-sentiment-analysis/data/main/cafef/cafef_input.parquet --max-tokens 256
2026-05-26 14:22:03,358 - INFO - Prepared 86179 CafeF rows -> /kaggle/working/news-sentiment-analysis/data/main/cafef/cafef_input.parquet
2026-05-26 14:22:03,385 - INFO - Wrote input preparation report -> /kaggle/working/news-sentiment-analysis/data/main/cafef/input_preparation_report.json
2026-05-26 14:22:05,234 - INFO - Running: /usr/bin/python3 -m src.sentiment.infer_cafef --model-dir /kaggle/working/news-sentiment-analysis/models/phobert-sentiment/latest --input-file /kaggle/working/news-sentiment-analysis/data/main/cafef/cafef_input.parquet --output-file /kaggle/working/news-sentiment-analysis/data/main/processed/article_sentiment_scores.parquet --batch-size 16 --max-length 256
2026-05-26 1

,url,trading_date,category,sentiment_score,sentiment_label,prob_positive,prob_negative,prob_neutral
0,https://cafef.vn/1-doanh-nghiep-o-quang-ninh-c...,2022-01-17,Doanh nghiệp,0.520954,positive,0.739460,0.218506,0.042034
1,https://cafef.vn/1-doanh-nghiep-tren-san-bi-bu...,2021-12-31,Doanh nghiệp,-0.992614,negative,0.003079,0.995693,0.001229
2,https://cafef.vn/1-can-bo-phong-tai-chinh-o-bi...,2018-11-19,Tài chính,-0.992291,negative,0.003156,0.995447,0.001398
3,https://cafef.vn/1-chuong-trinh-da-thay-doi-cu...,2024-12-05,Tài chính,0.992037,positive,0.992553,0.000516,0.006930
4,https://cafef.vn/1-dieu-ma-tat-ca-nguoi-giau-t...,2023-06-05,Kinh doanh,0.711315,positive,0.800110,0.088795,0.111094
